# 1. Load all dependencies and set up environment variables

In [ ]:
import dotenv

from quotaclimat.data_ingestion.factiva.utils_data_processing.detect_keywords import (
    create_combined_regex_pattern,
)
from quotaclimat.data_ingestion.factiva.utils_data_processing.utils_extract import (
    load_source_classification,
    poll_snapshot_explain,
    submit_snapshot_explain,
    get_account_statistics,
    submit_snapshot_extraction,
    poll_snapshot_extraction,
    download_snapshot_file,
    upload_snapshot_files_to_s3,
)

from quotaclimat.data_ingestion.factiva.factiva_to_s3.factiva_api_utils import (
    submit_time_series, poll_time_series, download_time_series_results)

from quotaclimat.data_processing.mediatree.keyword.keyword import THEME_KEYWORDS
from quotaclimat.data_ingestion.factiva.inputs.classification_source import SOURCE_CLASSIFICATION



# Make sure to set FACTIVA_USERKEY in the .env file at the root of the project
dotenv.load_dotenv()

# 2. Set up all extraction parameters

In [ ]:
START_DATE = "2025-01-01"
END_DATE = None
MINIMAL_WORD_COUNT = 0
LAUNCH_CREATE_SNAPSHOT_EXTRACT = False
LAUNCH_UPLOAD_TO_S3 = True

In [ ]:
# Create regex pattern for article filtering
keywords_filtered = []
for lst in THEME_KEYWORDS.values():
    for entry in lst:
        if (
            not entry.get("high_risk_of_false_positive", True)
            and entry.get("language") == "french"
        ):
            keywords_filtered.append(entry.get("keyword"))

# keywords_filtered contient la liste désirée
keywords_filtered = list(set(keywords_filtered))

keyword_regex = create_combined_regex_pattern(
    keywords_filtered, bigquery_compatible=True
)

In [ ]:
# Set up sources to extract
all_sources = load_source_classification(field_name='source_code', source_classification=SOURCE_CLASSIFICATION)

# 3. Verify the associated number of items to extract before launching a snapshot extraction

## The “snapshot explain” call can take a long time for large queries (several hours)

In [ ]:
submit_result = submit_snapshot_explain(
    source_codes=all_sources,
    start_date=START_DATE,
    end_date=END_DATE,
    minimal_word_count=0,
    language_code="fr",
    regex_pattern=keyword_regex,
)
submit_result

In [ ]:
if submit_result["success"]:
    explain_id = submit_result["explain_id"]
    print(f"Job submitted successfully! ID: {explain_id}")


    poll_result = poll_snapshot_explain(
        explain_id=explain_id,
        max_attempts=20,
        wait_seconds=30,
    )
    
    print(f"Polling result: {poll_result}")
else:
    print(f"Error during submission: {submit_result['error']}")

# 4. Launch Snapshot extract

## Warning: this action will consume Factiva credits. Please check the number of articles extracted beforehand.

## The “snapshot extract” can take a long time for large queries (several hours)

## 4.1 Create snapshot extraction

In [ ]:
if LAUNCH_CREATE_SNAPSHOT_EXTRACT:
    print("="*60)
    print("LAUNCHING SNAPSHOT EXTRACTION")
    print("="*60)
    
    extraction_result = submit_snapshot_extraction(
        source_codes=all_sources,
        start_date=START_DATE,
        end_date=END_DATE,
        minimal_word_count=MINIMAL_WORD_COUNT,
        language_code="fr",
        regex_pattern=keyword_regex,
        file_format="json",
        shards=None,  # Factiva decides (shards only works for >10M articles)
    )
    LAUNCH_CREATE_SNAPSHOT_EXTRACT = False

## 4.2 Monitor extraction status

In [ ]:
extraction_id = extraction_result["extraction_id"]
extraction_id

In [ ]:
poll_result = poll_snapshot_extraction(
    extraction_id=extraction_id,
    max_attempts=10,
    wait_seconds=20,
    extended=True,
)

In [ ]:
if poll_result["success"]:
    num_files = len(poll_result['files'])
    
    print("\n✅ Extraction completed successfully!")
    print(f"Factiva created: {num_files} files")
    
    if poll_result.get('article_count'):
        total_articles = poll_result['article_count']
        articles_per_file = total_articles // num_files
        print(f"Total articles: {total_articles:,}")
        print(f"Approximately {articles_per_file:,} articles per file")

## 4.3 Download locally extracted articles

In [ ]:
print("\n" + "="*60)
print(f"DOWNLOADING {num_files} FILES")
print("="*60)

download_dir = f"data/factiva_snapshots/{extraction_id}"
downloaded_files = []

for idx, file_info in enumerate(poll_result['files'], 1):
    file_uri = file_info['uri']
    file_name = f"factiva_raw_part_{idx:03d}.json.gz"
    output_path = f"{download_dir}/raw/{file_name}"
    
    print(f"\nDownloading file {idx}/{num_files}...")
    download_result = download_snapshot_file(
        file_uri=file_uri,
        output_path=output_path,
        use_stream_delivery=False,  # Set to True if Google domains are blocked
    )
    
    if download_result["success"]:
        downloaded_files.append(download_result["file_path"])
        print(f"  ✅ {file_name} ({download_result['file_size_mb']:.2f} MB)")
    else:
        print(f"  ❌ Error: {download_result['error']}")

print(f"\n✅ Downloaded {len(downloaded_files)}/{num_files} files")

## 4.4 Push extracted files to Scaleway S3

In [ ]:
FIRST_ARTICLE = 1
LAST_ARTICLE = 1
LAUNCH_UPLOAD_TO_S3 = True

if LAUNCH_UPLOAD_TO_S3:
    print("="*70)
    print("STARTING UPLOAD TO S3")
    print("="*70)
    
    file_dir = download_dir + "/raw"
    # Upload les fichiers vers S3
    upload_result = upload_snapshot_files_to_s3(
        downloaded_files_dir=file_dir,
        first_article=FIRST_ARTICLE,
        last_article=LAST_ARTICLE,
        delete_local_after_upload=False,  # Mettre à True pour supprimer les fichiers locaux après upload
    )
    
    if upload_result["success"]:
        print("\n✅ Upload completed successfully!")
        print(f"   Files uploaded: {len(upload_result['uploaded_files'])}")
        print(f"   Total articles: {upload_result['total_articles']:,}")
    else:
        print(f"\n❌ Upload failed: {upload_result['error']}")
    
    LAUNCH_UPLOAD_TO_S3 = False
else:
    print("Set LAUNCH_UPLOAD_TO_S3 = True to upload files to S3")

# 5. Manage account

## 5.1 Verify account statistics

In [ ]:
account_statistics = get_account_statistics()
account_statistics